In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

import pyarrow.parquet as pq

TOP_PER_CLASS = 5
acc = {}
for f in sorted(glob.glob(f"{SLIM_DIR}/*_slim.parquet")):
    names = pq.ParquetFile(f).schema.names
    genes = [g for g in GRMS if g in names]
    d = pd.read_parquet(f, columns=["class", "subclass"] + genes)
    g = d.groupby(["class", "subclass"])
    s = g[genes].sum()
    s["n"] = g.size()
    acc[f] = s
tot = pd.concat(acc.values()).groupby(level=[0, 1]).sum()
n = tot.pop("n")
mean = tot.div(n, axis=0)
mean["n_cells"] = n
mean = mean.reset_index()
mean.to_csv(f"{RES_DIR}/subclass_grm_mean_log2.csv", index=False)
m = mean[mean["n_cells"] >= MIN_CELLS].copy()
m["cn"] = m["class"].map(class_num)
genes = [g for g in GRMS if g in m.columns]
m["peak"] = m[genes].max(axis=1)
cond = m.sort_values("peak", ascending=False).groupby("class").head(TOP_PER_CLASS).sort_values(["cn", "subclass"]).reset_index(drop=True)
full = m.sort_values(["cn", "subclass"]).reset_index(drop=True)
vmax = float(np.percentile(m[genes].to_numpy(), 99.5))

def draw(tbl, per_row, title, fname):
    n = len(tbl)
    ns = int(np.ceil(n / per_row))
    SH, TOPM, BOTM, W = 4.8, 1.1, 1.4, 18
    H = ns * SH + TOPM + BOTM
    fig = plt.figure(figsize=(W, H))
    L, R = 0.06, 0.97
    heat_h, strip_h = 0.27 * len(genes), 0.18
    im = None
    for k in range(ns):
        ch = tbl.iloc[k * per_row:(k + 1) * per_row]
        w = (R - L) * len(ch) / per_row
        top = H - TOPM - k * SH
        ax_s = fig.add_axes([L, (top - 2.3 - strip_h) / H, w, strip_h / H])
        ax_h = fig.add_axes([L, (top - 2.3 - strip_h - heat_h) / H, w, heat_h / H])
        ax_s.imshow(np.array([class_color(c) for c in ch["class"]])[None, :, :], aspect="auto", interpolation="nearest")
        ax_s.set_xticks(range(len(ch)))
        ax_s.set_xticklabels(ch["subclass"], rotation=90, fontsize=7.5)
        ax_s.xaxis.tick_top()
        ax_s.tick_params(length=0, pad=2)
        ax_s.set_yticks([])
        im = ax_h.imshow(ch[genes].T.to_numpy(), aspect="auto", cmap="viridis", vmin=0, vmax=vmax, interpolation="nearest")
        ax_h.set_yticks(range(len(genes)))
        ax_h.set_yticklabels(genes, fontsize=10)
        ax_h.set_xticks([])
        for a_ in (ax_s, ax_h):
            for sp in a_.spines.values():
                sp.set_visible(False)
    cax = fig.add_axes([0.72, (H - 0.8) / H, 0.25, 0.13 / H])
    cb = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.ax.tick_params(labelsize=8)
    fig.text(0.72, (H - 0.55) / H, "mean log2(CPM+1)", fontsize=9)
    fig.text(L, (H - 0.2) / H, title, fontsize=11, va="top")
    cls_in = tbl.drop_duplicates("class").sort_values("cn")["class"].tolist()
    fig.legend(handles=[plt.Line2D([], [], marker="s", ls="", color=class_color(c), label=c, markersize=7) for c in cls_in],
               loc="lower center", ncol=6, fontsize=8, frameon=False, bbox_to_anchor=(0.5, 0.0), title="class (color strip above each column)", title_fontsize=9)
    fig.savefig(f"{FIG_DIR}/{fname}", dpi=170, bbox_inches="tight")
    plt.show()

draw(cond, 58, f"Whole mouse brain (Allen WMB, 10x; all regions pooled): mean log2(CPM+1) of Grm1-8 per subclass.\nShown: the {TOP_PER_CLASS} subclasses per class with the highest Grm expression ({len(cond)} of {len(full)} subclasses with >= {MIN_CELLS} cells).", "ALL_fig4_heatmap_subclass_grm.png")
draw(full, 60, f"Whole mouse brain (Allen WMB, 10x; all regions pooled): mean log2(CPM+1) of Grm1-8 for all {len(full)} subclasses with >= {MIN_CELLS} cells.", "ALL_fig4_heatmap_all_subclasses_full.png")